## Dataset

- Startup failures dataset (by Daglox Kanwanda): includes post-mortems for haundreds of startup with binary flags for seasons like poor market fit, no budget, and competition.

- Big startup success /fail dataset from crunchbase (by Yan Maksi): contains 60,000+ startups categorized by funding totals, regions industry sectors, and operational status (operating, closed, acquired)

1. Which combinations of runway, burn, and LTV/CAC distinguish high-risk startups from healthy ones across stages and categories?
Relevant dataset: synthetic_startup_metrics.csv
Diagnostic value: Identifies whether the immediate bottleneck is cash survival, weak unit economics, or both, and determines which KPI founders should address first.
2. Which failure factors commonly occur together within each sector, and how do they vary with funding raised and operating lifespan?
Relevant datasets: All six sector-specific Startup Failure datasets
Diagnostic value: Creates sector-specific failure archetypes and helps founders test the most relevant underlying risks instead of reacting only to visible symptoms.
3. Within comparable categories, regions, and founding cohorts, how are funding amount, number of rounds, and funding timing associated with observed startup outcomes?
Relevant dataset: big_startup_secsees_dataset.csv
Diagnostic value: Benchmarks a startup’s funding trajectory against peers and highlights possible undercapitalization, fundraising stagnation, or inefficient use of capital.

## Data-analysis pipeline
1. Define the questions and outcomes
Specify the three diagnostic questions, unit of analysis, comparison groups, and target variables.

2. Audit the raw data
Check schemas, row counts, missing values, duplicates, invalid values, class balance, and whether each source is historical or synthetic.

3. Clean and standardize
Normalize company names, sectors and categories; parse funding and dates; harmonize failure flags; validate runway and LTV/CAC calculations.

4. Build analysis-ready tables
Combine the six sector failure files vertically. Keep the failure, Crunchbase, and synthetic-metrics datasets separate unless company matches are verified.

5. Engineer useful features
Create company age, funding delay, funding span, funding bands, failure-factor counts, risk combinations, runway bands, and peer cohorts.

6. Perform exploratory analysis
Examine distributions, outliers, correlations, failure-factor co-occurrence, and differences across sectors, stages, regions, and outcomes.

7. Answer the three questions
    - Health-risk analysis using runway and unit economics.
    - Failure-archetype analysis using factor combinations and sectors.
    - Funding-outcome benchmarking using comparable startup cohorts.

8. Validate the findings
Use holdout testing or cross-validation, address class imbalance, test robustness, report uncertainty, and avoid causal claims from observational data.

9. Convert findings into diagnostics
Produce a ranked bottleneck, supporting evidence, possible causes, recommended test, and KPI for measuring the result.
Document and reproduce

10. Save cleaning rules, assumptions, data dictionaries, analysis code, quality checks, and final outputs so the pipeline can be rerun.

## Step 1 — Define the analytical contract
- Project objective: 

    Build a prototype diagnostic system that converts startup metrics and historical evidence into:

- Observed pattern → likely bottleneck → evidence-based
hypothesis → recommended test → measurement

- The system will rank diagnostic hypotheses. It will not claim to prove causes.

### Analysis questions
1. Which combinations of runway, burn, and unit economics distinguish high-risk startups from healthy startups?

- Unit of analysis: One startup
- Output: health_status: High Risk, Moderate Risk, Healthy
- Main comparision: Startups at the same stage and in the same category

2. Which failure factors commonly occur together, and how do these patterns differ by sector, funding raised, and operating lifespan?

- Unit of analysis: One failed startup
- Output: Binary failure factors and derived failure archetype
- Main comparision: Sector, funding band, and lifespan band

3. Among comparable startups, how are funding amount, rounds, and timing associated with observed company status?
- Unit of analysis: individual company
- Output: status: operating, closed, acquired, or IPO
- Main comparision: Category, geography, and founding cohort


### Dataset assignment
1. Startup health analysis
- Dataset: synthetic_startup_metrics.csv
- Inputs: MRR, CAC, LTV, LTV/CAC, burn, cash, runway, stage, and category.
- Intended output: health classification, peer percentile, and dominant risk driver.

2. Failure-archetype analysis
- Datasets: the six detailed sector-specific Startup Failure files.
- Startup Failures.csv will support coverage and consistency checks.
- Intended output: Sector-specific prevalence of individual failure factors, meaningful co-occurring failure combinations, and ranked primary and contributing failure hypotheses.

3. Funding-outcome analysis
- Dataset: big_startup_secsees_dataset.csv
- Inputs: funding total, funding rounds, funding dates, category, geography, and founding date.
- Intended output: peer benchmark and association with each observed status.

### Interpretation rules
- The three datasets will remain separate unless a company match is verified.
- Synthetic metrics will support prototype development, not real-world validation.
- Failure data describe patterns among failed companies; they cannot directly estimate a startup’s probability of failure.
- Funding relationships will be reported as associations, not causal effects.
- The four Crunchbase statuses will initially remain separate.
- Only failure flags consistently defined across sectors will be used in cross-sector comparisons. Sector-specific flags will be analysed separately.

## Step 2 — Data-quality audit

The script checks:

- Dataset dimensions and data types
- Missing values
- Duplicate rows and identifiers
- Invalid binary failure flags
- Funding and status validity
- Funding-date order
- Negative operating metrics
- LTV/CAC and runway calculation consistency

_Command:_ 
- python -m pip install pandas
- python startup_data_quality_audit.py

_Short audit note_
Overall: The datasets are structurally usable and contain no major validity errors.
Big startup dataset: The main concern is missing data, especially founded_at and funding_total_usd. The 330 repeated names should be investigated, not automatically deleted, because different companies can share a name.
Failure datasets: They are nearly complete. Only one Overhype value is missing. The master failure file has one duplicate row.
Synthetic metrics: The dataset is clean, internally consistent, and ready for analysis.

The next step will be deciding how to handle each missing value, duplicate, and inconsistent schema before cleaning the data.

## Step 3 — Clean and standardize
The script:

- Normalizes company names without deleting repeated names.
- Removes exact duplicates and duplicate unique IDs.
- Parses funding amounts and dates.
- Marks uncertain funding strings for review.
- Standardizes categories, locations, sectors, and statuses.
- Gives all sector files the same failure-flag columns.
- Treats unavailable flags as missing—not as zero.
- Keeps unknown numeric values and dates missing instead of inventing values.
- Recalculates and validates LTV/CAC and runway.
- Preserves all original CSV files.

Testing confirmed that the one duplicate master failure record was removed and all 200 synthetic records passed the LTV/CAC and runway validation checks. Step 4 has not started.

## Step 4.
It writes these files to data/processed:

- failure_analysis_table.csv — 409 combined sector-failure records
- crunchbase_analysis_table.csv — 66,368 companies, kept separate
- startup_metrics_analysis_table.csv — 200 validated records, kept separate
- potential_company_matches.csv — possible matches requiring manual verification
- analysis_table_summary.csv — output dimensions and purposes

The master failure dataset remains separate because it does not contain detailed failure flags. No datasets are automatically joined by company name.

## Step 5.
This notebook creates only the practical features needed for the three analysis
questions:

- Company age
- Funding delay and funding span
- Funding bands
- Failure-factor counts and risk combinations
- Runway bands
- Peer cohorts

The three datasets remain separate. The Step 4 tables are read from
`data/processed`, and new feature tables are saved back to that folder.

## 6. Perform exploratory analysis

This is the single EDA notebook for the project. It replaces the earlier
exploration notebooks and uses only the three feature tables created in Step 5.

The notebook examines:

- Distributions and outliers
- Correlations
- Failure-factor co-occurrence
- Differences across sectors and stages
- Differences across regions and observed company outcomes

It does not clean data, create predictive models, or merge the three datasets.


## Step 7:



Main answers:

- Health risk: Runway is the most common immediate bottleneck—93 runway breaches versus 63 unit-economics breaches; 28 breach both.
- Failure archetype: Large incumbents and competition occur together in 58.2% of historical failure records. Sector-specific secondary risks determine the recommended next test.
- Funding benchmark: Across 97 reliable peer cohorts, observed exit share rises from 7.5% in the bottom funding quarter to 26.0% in the top quarter.

The notebook contains simple commented code, three readable charts, result notes, recommended investigations, and saved diagnostic tables.

## Step 8.
The notebook includes:

- Holdout validation
- Class-imbalance handling
- Bootstrap 95% confidence intervals
- Threshold and cohort-size robustness tests
- Clear result notes under each analysis
- Explicit warnings against causal interpretation
- Executed outputs and readable graphs

## Step 9
The notebook:

- Produces ranked bottlenecks with evidence, possible causes, tests, KPIs, and success signals.
- Creates separate startup-health, sector-risk, and funding diagnostic tables.
- Documents all thresholds and ranking rules.
- Includes reproducibility checks and a run manifest.
- Keeps unmatched datasets separate.
- Ran successfully without errors.

## step 10

The package contains:

- Cleaning rules and analysis assumptions
- Data dictionary
- Raw, cleaned, and processed datasets
- Steps 4–10 notebooks
- Data-quality checks
- Code and data manifests with file hashes
- Final tables, charts, diagnostics, and validation outputs
- Environment versions and requirements
- Pipeline README
- One-command pipeline runner

To reproduce everything from the project folder:

_python src/run_pipeline.py_

This executes the pipeline in order:

- Data-quality audit
- Cleaning and standardization
- Build analysis-ready tables
- Engineer features
- Exploratory analysis
- Answer the three questions
- Validate findings
- Generate startup diagnostics
- Save documentation and final checks

You can also restart from a particular analysis step:

python src/run_pipeline.py --from-step 6

The ZIP is a portable copy of the complete project, containing:

- Raw, cleaned, and processed data
- All analysis notebooks
- Python scripts and the pipeline runner
- Documentation and data dictionaries
- Quality-check results
- Graphs, tables, validation results, and final diagnostics
- Requirements needed to recreate the environment

The ZIP itself does not run anything. Extract it first, open the extracted project folder, and run:

python src/run_pipeline.py

pip install -r requirements.txt